In [4]:
from sklearn.metrics import accuracy_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor, Compose, Normalize
from torchvision.transforms import ConvertImageDtype, Resize
from matplotlib import pyplot as plt
from tqdm import tqdm
from pathlib import Path


import torch
import numpy as np
import pandas as pd
import os

In [7]:
DATA_DIR = Path('../data/10monkeys/archive')
IMG_SIZE = (224, 224)


class MonkeyDataset(datasets.ImageFolder):
    def __init__(self, mode, transform=None):

        root = DATA_DIR / "training" if mode == 'train' else DATA_DIR / "validation"

        super().__init__(root=root, transform=transform)
        self.targets = [_[1] for _ in self.samples]


transform = Compose([
    Resize(IMG_SIZE),
    ToTensor(),
    ConvertImageDtype(torch.float32)
    # 计算出 mean/std 后改成：Normalize(MONKEY_MEAN, MONKEY_STD)
])

In [9]:
def make_image_transform(img_size, mean=None, std=None, dtype=torch.float32):
    """通用图像变换函数：可选是否执行 Normalize。"""
    tfs = [
        Resize(img_size),
        ToTensor(),
    ]
    if mean is not None and std is not None:
        tfs.append(Normalize(mean, std))
    tfs.append(ConvertImageDtype(dtype))
    return Compose(tfs)


def compute_dataset_mean_std(dataset, batch_size=64, num_workers=0):
    """通用统计函数：计算任意图像分类数据集的每通道 mean/std。"""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    channel_sum = None
    channel_sq_sum = None
    pixel_count = 0

    for imgs, _ in loader:
        if channel_sum is None:
            c = imgs.size(1)
            channel_sum = torch.zeros(c, dtype=torch.float64)
            channel_sq_sum = torch.zeros(c, dtype=torch.float64)

        imgs = imgs.to(torch.float64)
        channel_sum += imgs.sum(dim=[0, 2, 3])
        channel_sq_sum += (imgs ** 2).sum(dim=[0, 2, 3])
        pixel_count += imgs.size(0) * imgs.size(2) * imgs.size(3)

    mean = channel_sum / pixel_count
    std = torch.sqrt(channel_sq_sum / pixel_count - mean ** 2)
    return tuple(mean.tolist()), tuple(std.tolist())


# 1) 用不带 Normalize 的 transform 统计训练集参数
stat_transform = make_image_transform(IMG_SIZE)
train_stat_ds = MonkeyDataset('train', transform=stat_transform)
MONKEY_MEAN, MONKEY_STD = compute_dataset_mean_std(train_stat_ds, batch_size=64, num_workers=0)

print('MONKEY_MEAN =', MONKEY_MEAN)
print('MONKEY_STD  =', MONKEY_STD)

# 2) 构建真正训练时使用的标准化 transform
transform = make_image_transform(IMG_SIZE, MONKEY_MEAN, MONKEY_STD)
print('transform 已更新为带 Normalize 的版本。')

MONKEY_MEAN = (0.4363363339137749, 0.4328124737860982, 0.32910693478250885)
MONKEY_STD  = (0.24640002027397762, 0.24186058305549507, 0.24529236291725595)
transform 已更新为带 Normalize 的版本。
